# 5. GPT-2 inference with metrics


In [1]:
# Dynamicly load evaluation metrics and the model
%run -i ../src/models/evaluation.py
%run -i ../src/models/detoxGPT2.py

In [2]:
import numpy as np
import pandas as pd
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
detoxGPT = detoxGPT2()

# Instantiate metric classes
similarity = Similarity()
toxicity = STAToxic()

In [4]:
prompt = "What a fucking stupid thing to say!"

In [ ]:
# Pack the suggestions into a dataframe
df = pd.DataFrame(
    detoxGPT.get_detoxed_suggestions(prompt, max_length=len(prompt), device=DEVICE), columns=["suggestion"]
)

# Add empty column for each metric
metrics = ["wo", "cs", "bleu"]
df[metrics] = pd.DataFrame([[0] * len(metrics)], index=df.index, dtype=float)

# Generate toxicity report for each suggestion
toxicity_report = toxicity.toxicity_report(df["suggestion"])
toxicity_report_columns = toxicity_report.columns

for index, row in df.iterrows():
    df.loc[index, "wo"] = similarity.get_wo_score(prompt, row["suggestion"])
    df.loc[index, "cs"] = similarity.get_cosine_score(prompt, row["suggestion"])
    df.loc[index, "bleu"] = similarity.get_bleu_score(prompt, row["suggestion"])

# Concat with toxicity report
df = pd.concat([df, toxicity_report], axis=1)
df

In [6]:
# Calculate the score
metric_weights = {"wo": 0.1, "cs": 0.5, "bleu": 0.4}

# Toxicity report should be as low as possible
# Similarity metrics should be as high as possible
df["detox_score"] = 1 - np.mean(df[["toxic"]], axis=1)
df["similarity"] = df[metrics].dot(pd.Series(metric_weights))

# Final score
df["score"] = df[["detox_score", "similarity"]].mean(axis=1)

# Sort by score
df = df.sort_values(by=["score"], ascending=False)
df

,suggestion,wo,cs,bleu,toxic,severe_toxic,obscene,threat,insult,identity_hate,detox_score,similarity,score
6,what a dreadful thing to say!,0.444444,0.972794,5.234257e-01,0.013593,0.001437,0.005581,0.000121,0.006205,0.000841,0.986407,0.740212,0.863309
5,what a terrible thing to say!,0.444444,0.972815,5.106913e-01,0.131332,0.000517,0.007929,0.000122,0.012770,0.001130,0.868668,0.735128,0.801898
4,what an awful thing to say to you about that!,0.333333,0.856849,3.796096e-01,0.260261,0.016070,0.038225,0.000100,0.070889,0.002337,0.739739,0.613602,0.676670
1,It is terrible to say too,0.181818,0.769838,2.031266e-01,0.257702,0.000450,0.007336,0.000074,0.014466,0.000647,0.742298,0.484352,0.613325
7,well,0.000000,0.325014,0.000000e+00,0.037838,0.001227,0.005018,0.000238,0.006115,0.000733,0.962162,0.162507,0.562335
2,Oh!,0.000000,0.410773,3.837170e-236,0.209103,0.003306,0.037315,0.000104,0.023747,0.009105,0.790897,0.205386,0.498142
0,"oh, my God!”",0.000000,0.501184,2.342016e-232,0.268023,0.002160,0.031497,0.000104,0.018096,0.001659,0.731977,0.250592,0.491285
3,Oh! Oh,0.000000,0.363762,1.310375e-233,0.209103,0.003306,0.037315,0.000104,0.023747,0.009105,0.790897,0.181881,0.486389


In [7]:
# Print the suggestion with the highest score
suggestion = df.iloc[0]["suggestion"]
print(f"{prompt} -> {suggestion}")

What a fucking stupid thing to say! -> what a dreadful thing to say!
